# 06 · Label-correlation model

The flat model's linear head is replaced by 39 learned label queries and a
two-layer transformer decoder (d = 512, 8 heads). Each query cross-attends to the
Swin patch tokens and self-attends to every other label, so evidence for one
disease can raise or lower the score of a co-occurring one. Everything else is
identical to the flat baseline.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # or: pip install -e ..

import numpy as np, pandas as pd, torch
from fundus import config, labels
from fundus.data import (load_split, unhealthy_only, FundusDataset, make_loader,
                         train_transform, eval_transform, pos_weight)
from fundus.engine import set_seed, device, fit, predict

P = config.Paths()          # reads $FUNDUS_DATA_ROOT and $FUNDUS_RUN_ROOT
set_seed(config.SEED)
DEV = device()
print("device:", DEV, "| data:", P.data_root.resolve())

## Co-occurrence that an independent sigmoid head cannot represent

In [ ]:
TG = labels.TARGETS
tr = load_split(P.csv("train"))
M = tr[TG].values
co = M.T @ M
pairs = sorted(((TG[i], TG[j], int(co[i, j])) for i in range(len(TG)) for j in range(i + 1, len(TG))
                if co[i, j] > 0), key=lambda z: -z[2])
print(f"{len(pairs)} disease pairs co-occur at least once; top 10:")
for a, b, n in pairs[:10]:
    print(f"  {a:5s} {b:5s} {n:4d}")

In [ ]:
from fundus.models import LabelCorrelationNet
from fundus.losses import make_criterion
from fundus.engine import evaluate_multilabel

cfg = config.LABEL_CORR
va = load_split(P.csv("val"))
train_ds = FundusDataset(tr, P.images("train"), train_transform(cfg.img_size), label_cols=TG)
val_ds   = FundusDataset(va, P.images("val"),   eval_transform(cfg.img_size),  label_cols=TG)
train_dl = make_loader(train_ds, cfg.batch_size, True,  cfg.num_workers)
val_dl   = make_loader(val_ds,   cfg.batch_size, False, cfg.num_workers)

model = LabelCorrelationNet(len(TG), cfg)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
criterion = make_criterion("bce", pos_weight(train_ds.labels, cfg.pos_weight_clip))
OUT = P.run("label_correlation")
hist = fit(model, train_dl, val_dl, criterion, evaluate_multilabel, OUT, cfg, monitors=("macro_ap",))